In [20]:
import pandas as pd
import numpy as np
import scipy.stats as stats
from scipy.stats import chi2_contingency, ttest_ind, f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load data
df = pd.read_csv('../data/insurance_data_cleaned.csv')

print("Data loaded successfully")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Data loaded successfully
Shape: (10000, 21)
Columns: ['CustomerID', 'Age', 'Gender', 'Province', 'VehicleType', 'AnnualIncome', 'RiskScore', 'AnnualPremium', 'Deductible', 'NCD', 'PastClaims', 'Claimed', 'ClaimAmount', 'TotalPremium', 'TotalClaims', 'CoverType', 'AutoMake', 'VehicleModel', 'CustomValueEstimate', 'ZipCode', 'TransactionDate']


In [19]:
df.head(5)

AttributeError: 'Index' object has no attribute '_format_flat'

  CustomerID  Age  Gender     Province VehicleType  AnnualIncome  RiskScore  \
0  AC-100000   56    Male  Addis Ababa       Sedan        147270         61   
1  AC-100001   69  Female  Addis Ababa         SUV         74640         57   
2  AC-100002   46    Male       Oromia       Sedan         70555         42   
3  AC-100003   32  Female       Somali       Sedan         89398         63   
4  AC-100004   60  Female       Tigray         SUV         78475         69   

   AnnualPremium  Deductible  NCD  ...  TotalClaims                 CoverType  \
0           2346         500   30  ...          0.0             Comprehensive   
1           2334         500    0  ...       9883.0             Comprehensive   
2           1697         250   20  ...          0.0  Third Party Fire & Theft   
3           2370         500   20  ...      12134.0             Comprehensive   
4           2582         500    0  ...          0.0             Comprehensive   

   AutoMake  VehicleModel  CustomValue

In [22]:
# Create metrics needed for hypothesis testing
df['Margin'] = df['TotalPremium'] - df['TotalClaims']
df['LossRatio'] = df['TotalClaims'] / df['TotalPremium']

# For claim severity (only policies WITH claims)
claims_data = df[df['Claimed'] == True].copy()

print("Metrics created:")
print(f"Total policies: {len(df):,}")
print(f"Policies with claims: {len(claims_data):,} ({len(claims_data)/len(df)*100:.1f}%)")
print(f"Policies without claims: {len(df) - len(claims_data):,}")

Metrics created:
Total policies: 10,000
Policies with claims: 1,535 (15.3%)
Policies without claims: 8,465


## Task 3: KPI Selection Summary

### Hypothesis 1: No risk differences across provinces
**KPI Selected:** Claim Frequency AND Claim Severity  
**Number of Tests:** 2  
**Reason:** Risk has two components — frequency (how often claims occur) and severity (how large claims are when they occur)

---

### Hypothesis 2: No risk differences between zip codes
**KPI Selected:** Claim Frequency AND Claim Severity  
**Number of Tests:** 2  
**Reason:** Same logic as provinces — both aspects of risk matter for granular geographic analysis

---

### Hypothesis 3: No significant margin difference between zip codes
**KPI Selected:** Margin ONLY  
**Number of Tests:** 1  
**Reason:** Margin is the bottom-line profitability metric (`TotalPremium - TotalClaims`)

---

### Hypothesis 4: No significant risk difference between Women and Men
**KPI Selected:** Claim Frequency AND Claim Severity  
**Number of Tests:** 2  
**Reason:** Gender could affect both frequency (likelihood of filing a claim) and severity (average claim amount)

---

## Summary Table

| Hypothesis | KPI | Tests |
|------------|-----|-------|
| H₁: Provinces | Claim Frequency + Claim Severity | 2 |
| H₂: Zip Codes | Claim Frequency + Claim Severity | 2 |
| H₃: Zip Codes (Margin) | Margin | 1 |
| H₄: Gender | Claim Frequency + Claim Severity | 2 |

**Total statistical tests to run: 7**

In [36]:

print( "Provincial Risk Differences")
# Create subset for Amhara and Somali only
province_subset = df[df['Province'].isin(['Amhara', 'Somali'])] #checks if a given row contains provinces Amhara and Somalia

print(" Count claims by province")

# Get counts using simple filtering
amhara_claimed = len(province_subset.loc[(province_subset['Province'] == 'Amhara') & (province_subset['Claimed'] == True)])
amhara_not_claimed = len(province_subset.loc[(province_subset['Province'] == 'Amhara') & (province_subset['Claimed'] == False)])
somali_claimed = len(province_subset.loc[(province_subset['Province'] == 'Somali') & (province_subset['Claimed'] == True)])
somali_not_claimed = len(province_subset.loc[(province_subset['Province'] == 'Somali') & (province_subset['Claimed'] == False)])

print(f"Amhara: {amhara_claimed} claims, {amhara_not_claimed} no claims")
print(f"Somali: {somali_claimed} claims, {somali_not_claimed} no claims")

# Create contingency table as list of lists
contingency = [
    [amhara_claimed, amhara_not_claimed],
    [somali_claimed, somali_not_claimed]
]

print(f"\nStep 2: Contingency Table")
print("-"*40)
print("            Claimed  Not Claimed")
print(f"Amhara      {contingency[0][0]:6}  {contingency[0][1]:6}")
print(f"Somali      {contingency[1][0]:6}  {contingency[1][1]:6}")

# Calculate frequencies
amhara_freq = amhara_claimed / (amhara_claimed + amhara_not_claimed) * 100
somali_freq = somali_claimed / (somali_claimed + somali_not_claimed) * 100

print(f"\nStep 3: Claim Frequencies")
print("-"*40)
print(f"Amhara claim frequency: {amhara_freq:.2f}%")
print(f"Somali claim frequency: {somali_freq:.2f}%")
print(f"Difference: {abs(amhara_freq - somali_freq):.2f} percentage points")

# Run chi-square test
chi2, p_value_freq, dof, expected = chi2_contingency(contingency)

print(f"\nStep 4: Chi-Square Test Results")
print(f"Chi-square statistic: {chi2:.4f}")
print(f"p-value: {p_value_freq:.4f}")
print(f"Degrees of freedom: {dof}")

# Decision
alpha = 0.05
if p_value_freq < alpha:
    print(f"REJECT H₀ (p={p_value_freq:.4f} < {alpha})")
    print("  → There IS a significant difference in claim frequency between Amhara and Somali")
else:
    print(f"\n✗ DECISION: FAIL TO REJECT H₀ (p={p_value_freq:.4f} ≥ {alpha})")
    print("  → No evidence of difference in claim frequency")

Provincial Risk Differences
 Count claims by province
Amhara: 279 claims, 1720 no claims
Somali: 207 claims, 977 no claims

Step 2: Contingency Table
----------------------------------------
            Claimed  Not Claimed
Amhara         279    1720
Somali         207     977

Step 3: Claim Frequencies
----------------------------------------
Amhara claim frequency: 13.96%
Somali claim frequency: 17.48%
Difference: 3.53 percentage points

Step 4: Chi-Square Test Results
Chi-square statistic: 6.8763
p-value: 0.0087
Degrees of freedom: 1
REJECT H₀ (p=0.0087 < 0.05)
  → There IS a significant difference in claim frequency between Amhara and Somali


In [33]:
# Get claim amounts for policies WITH claims only
claims_subset = province_subset[province_subset['Claimed'] == True]

# Get severity for each province
amhara_severity = claims_subset[claims_subset['Province'] == 'Amhara']['TotalClaims']
somali_severity = claims_subset[claims_subset['Province'] == 'Somali']['TotalClaims']

print(f"Amhara claims: n={len(amhara_severity)}, mean=R{amhara_severity.mean():.2f}")
print(f"Somali claims: n={len(somali_severity)}, mean=R{somali_severity.mean():.2f}")

# Run t-test (independent, since different groups)
t_stat, p_value = ttest_ind(amhara_severity, somali_severity)

print(f"\nt-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.4f}")

# Decision
if p_value < alpha:
    print("\nDecision: REJECT H₀ - There IS a significant difference in claim severity")
else:
    print("\nDecision: FAIL TO REJECT H₀ - No evidence of difference in claim severity")

Amhara claims: n=279, mean=R8436.86
Somali claims: n=207, mean=R8824.12

t-statistic: -0.6601
p-value: 0.5095

Decision: FAIL TO REJECT H₀ - No evidence of difference in claim severity


### Hypothesis 1: Provincial Risk Differences

**Finding:** We reject the null hypothesis for claim frequency (p=0.0087) but fail to reject for claim severity (p=0.5095).

**Interpretation:** 
- Somali province has a **17.48% claim frequency** compared to Amhara's 13.96% – a 3.53 percentage point difference that is statistically significant.
- However, when claims do occur, the average claim amount in Somali (R8,824) is not significantly different from Amhara (R8,437).

**Business Recommendation:**
ACIS should consider a **frequency-based risk adjustment** for Somali province. Since Somali policies are 25% more likely to file a claim (17.48% vs 13.96%), but claim amounts are similar, the appropriate action is to:
1. Increase premiums in Somali province to reflect higher claim frequency
2. OR implement targeted loss prevention programs in Somali (e.g., driver education, telematics)
3. Maintain current severity assumptions as they are not significantly different

In [38]:
print("ZIP CODE ANALYSIS - Selecting Control and Test Groups")
# Calculate loss ratio by zip code (only zip codes with enough data)
zip_stats = df.groupby('ZipCode').agg({
    'TotalPremium': 'sum',
    'TotalClaims': 'sum',
    'Claimed': 'count'
}).rename(columns={'Claimed': 'PolicyCount'})

# Calculate loss ratio
zip_stats['LossRatio'] = (zip_stats['TotalClaims'] / zip_stats['TotalPremium']) * 100

# Filter zip codes with at least 30 policies (for statistical validity)
zip_stats = zip_stats[zip_stats['PolicyCount'] >= 30]

print(f"\nZip codes with sufficient data (≥30 policies): {len(zip_stats)}")

print("\nTop 5 LOWEST risk zip codes (best for ACIS):")
print(zip_stats.nsmallest(5, 'LossRatio')[['PolicyCount', 'LossRatio']])

print("\nTop 5 HIGHEST risk zip codes (worst for ACIS):")
print(zip_stats.nlargest(5, 'LossRatio')[['PolicyCount', 'LossRatio']])

ZIP CODE ANALYSIS - Selecting Control and Test Groups

Zip codes with sufficient data (≥30 policies): 25

Top 5 LOWEST risk zip codes (best for ACIS):
         PolicyCount  LossRatio
ZipCode                        
30002            406  32.813256
10003            714  38.393054
50002            149  42.123129
20001            477  42.368006
50001            152  44.131443

Top 5 HIGHEST risk zip codes (worst for ACIS):
         PolicyCount  LossRatio
ZipCode                        
50004            154  73.145919
40004            238  68.227997
40001            234  67.226682
40005            193  64.691343
20002            479  62.360972


In [ ]:
print("HYPOTHESIS 2 & 3: Zip Code Risk Differences")
# Select control and test zip codes
control_zip = 30002
test_zip = 50004

# Create subset
zip_subset = df[df['ZipCode'].isin([control_zip, test_zip])]

print(f"Control Group (Lowest Risk): Zip Code {control_zip}")
print(f"Test Group (Highest Risk): Zip Code {test_zip}")
print(f"Total policies in comparison: {len(zip_subset)}")

# HYPOTHESIS 2: Claim Frequency (Chi-Square)

print("HYPOTHESIS 2: Claim Frequency Difference")


# Get counts
control_claimed = len(zip_subset[(zip_subset['ZipCode'] == control_zip) & (zip_subset['Claimed'] == True)])
control_not_claimed = len(zip_subset[(zip_subset['ZipCode'] == control_zip) & (zip_subset['Claimed'] == False)])
test_claimed = len(zip_subset[(zip_subset['ZipCode'] == test_zip) & (zip_subset['Claimed'] == True)])
test_not_claimed = len(zip_subset[(zip_subset['ZipCode'] == test_zip) & (zip_subset['Claimed'] == False)])

# Create contingency table
contingency_zip = [
    [control_claimed, control_not_claimed],
    [test_claimed, test_not_claimed]
]

print(f"\nContingency Table:")
print(f"            Claimed  Not Claimed")
print(f"Zip {control_zip}     {contingency_zip[0][0]:6}  {contingency_zip[0][1]:6}")
print(f"Zip {test_zip}     {contingency_zip[1][0]:6}  {contingency_zip[1][1]:6}")

# Calculate frequencies
control_freq = control_claimed / (control_claimed + control_not_claimed) * 100
test_freq = test_claimed / (test_claimed + test_not_claimed) * 100

print(f"\nClaim Frequency:")
print(f"  Zip {control_zip}: {control_freq:.2f}%")
print(f"  Zip {test_zip}: {test_freq:.2f}%")
print(f"  Difference: {abs(control_freq - test_freq):.2f} percentage points")

# Chi-square test
chi2, p_value_freq_zip, dof, expected = chi2_contingency(contingency_zip)

print(f"\nChi-Square Test Results:")
print(f"  Chi-square statistic: {chi2:.4f}")
print(f"  p-value: {p_value_freq_zip:.4f}")

if p_value_freq_zip < 0.05:
    print(f"REJECT H₀ - Significant difference in claim frequency between zip codes")
else:
    print(f"FAIL TO REJECT H₀ - No significant difference")


# HYPOTHESIS 2: Claim Severity (T-Test)
print("HYPOTHESIS 2: Claim Severity Difference")

# Get claims data
claims_zip = zip_subset[zip_subset['Claimed'] == True]

control_severity = claims_zip[claims_zip['ZipCode'] == control_zip]['TotalClaims'].values
test_severity = claims_zip[claims_zip['ZipCode'] == test_zip]['TotalClaims'].values

print(f"\nSeverity Statistics:")
print(f"  Zip {control_zip}: n={len(control_severity)}, Mean=R{control_severity.mean():.2f}, Median=R{np.median(control_severity):.2f}")
print(f"  Zip {test_zip}: n={len(test_severity)}, Mean=R{test_severity.mean():.2f}, Median=R{np.median(test_severity):.2f}")

# T-test
t_stat, p_value_sev_zip = ttest_ind(control_severity, test_severity)

print(f"\nT-Test Results:")
print(f"  t-statistic: {t_stat:.4f}")
print(f"  p-value: {p_value_sev_zip:.4f}")

if p_value_sev_zip < 0.05:
    print(f"REJECT H₀ - Significant difference in claim severity between zip codes")
else:
    print(f"FAIL TO REJECT H₀ - No significant difference")

# HYPOTHESIS 3: Margin Difference (T-Test)

print("HYPOTHESIS 3: Margin (Profit) Difference")

# Get margin values
control_margin = zip_subset[zip_subset['ZipCode'] == control_zip]['Margin'].values
test_margin = zip_subset[zip_subset['ZipCode'] == test_zip]['Margin'].values

print(f"Margin Statistics:")
print(f"  Zip {control_zip}: Mean=R{control_margin.mean():.2f}, Median=R{np.median(control_margin):.2f}")
print(f"  Zip {test_zip}: Mean=R{test_margin.mean():.2f}, Median=R{np.median(test_margin):.2f}")

# T-test
t_stat, p_value_margin = ttest_ind(control_margin, test_margin)

print(f"T-Test Results:")
print(f"  t-statistic: {t_stat:.4f}")
print(f"  p-value: {p_value_margin:.4f}")

if p_value_margin < 0.05:
    print(f" REJECT H₀ - Significant difference in margin between zip codes")
else:
    print(f" FAIL TO REJECT H₀ - No significant difference")

HYPOTHESIS 2 & 3: Zip Code Risk Differences
Control Group (Lowest Risk): Zip Code 30002
Test Group (Highest Risk): Zip Code 50004
Total policies in comparison: 560
HYPOTHESIS 2: Claim Frequency Difference

Contingency Table:
            Claimed  Not Claimed
Zip 30002         41     365
Zip 50004         26     128

Claim Frequency:
  Zip 30002: 10.10%
  Zip 50004: 16.88%
  Difference: 6.78 percentage points

Chi-Square Test Results:
  Chi-square statistic: 4.2565
  p-value: 0.0391

  ✓ REJECT H₀ - Significant difference in claim frequency between zip codes
HYPOTHESIS 2: Claim Severity Difference

Severity Statistics:
  Zip 30002: n=41, Mean=R7858.41, Median=R7450.00
  Zip 50004: n=26, Mean=R10774.00, Median=R6516.50

T-Test Results:
  t-statistic: -1.6775
  p-value: 0.0983

  ✗ FAIL TO REJECT H₀ - No significant difference
HYPOTHESIS 3: Margin (Profit) Difference
Margin Statistics:
  Zip 30002: Mean=R1624.90, Median=R2160.50
  Zip 50004: Mean=R667.81, Median=R2121.00
T-Test Results:
  

### Hypothesis 2: Zip Code Risk Differences

| Metric | Zip 30002 (Low Risk) | Zip 50004 (High Risk) | p-value | Decision |
|--------|---------------------|----------------------|---------|----------|
| Claim Frequency | 10.10% | 16.88% | 0.0391 | **REJECT H₀** |
| Claim Severity | R7,858 | R10,774 | 0.0983 | FAIL TO REJECT H₀ |

**Interpretation:** Zip code 50004 has a significantly higher claim frequency (16.88%) than zip code 30002 (10.10%) – a 6.78 percentage point difference (67% higher). While claim severity is 37% higher in zip 50004 (R10,774 vs R7,858), this difference is not statistically significant at α=0.05 (p=0.0983).

**Business Recommendation:** 
- **Immediate action:** Increase premiums for zip code 50004 by 15-20% to reflect higher claim frequency
- **Further investigation:** Monitor severity trend in zip 50004 – p-value near significance (0.0983) suggests possible real difference with more data
- **Competitive opportunity:** Zip code 30002 represents a low-risk segment – consider targeted marketing campaigns there with competitive premiums

### Hypothesis 3: Margin Difference Between Zip Codes

| Metric | Zip 30002 (Low Risk) | Zip 50004 (High Risk) | p-value | Decision |
|--------|---------------------|----------------------|---------|----------|
| Margin (Profit) | R1,625 | R668 | 0.0056 | **REJECT H₀** |

**Interpretation:** Zip code 30002 generates significantly higher profit per policy (R1,625) compared to zip code 50004 (R668) – a difference of R957 per policy. This represents a 143% higher profit margin in the low-risk zip code.

**Business Recommendation:**
- **Reallocate marketing budget:** Shift 60% of zip code 50004 marketing spend to zip code 30002 and similar low-risk areas
- **Premium restructuring:** Current premiums in zip 50004 are insufficient to cover the elevated risk – recommend 25-30% premium increase
- **Customer acquisition:** Launch targeted acquisition campaign in zip 30002 offering 10-15% discount to attract more low-risk customers

In [42]:

print("HYPOTHESIS 4: Gender Risk Differences")
print("Claim Frequency by Gender")

# Get counts
female_claimed = len(df[(df['Gender'] == 'Female') & (df['Claimed'] == True)])
female_not_claimed = len(df[(df['Gender'] == 'Female') & (df['Claimed'] == False)])
male_claimed = len(df[(df['Gender'] == 'Male') & (df['Claimed'] == True)])
male_not_claimed = len(df[(df['Gender'] == 'Male') & (df['Claimed'] == False)])

# Create contingency table
contingency_gender = [
    [female_claimed, female_not_claimed],
    [male_claimed, male_not_claimed]
]

print(f"\nContingency Table:")
print(f"            Claimed  Not Claimed")
print(f"Female      {contingency_gender[0][0]:6}  {contingency_gender[0][1]:6}")
print(f"Male        {contingency_gender[1][0]:6}  {contingency_gender[1][1]:6}")

# Calculate frequencies
female_freq = female_claimed / (female_claimed + female_not_claimed) * 100
male_freq = male_claimed / (male_claimed + male_not_claimed) * 100

print(f"\nClaim Frequency:")
print(f"  Female: {female_freq:.2f}%")
print(f"  Male: {male_freq:.2f}%")
print(f"  Difference: {abs(female_freq - male_freq):.2f} percentage points")

# Chi-square test
chi2, p_value_freq_gender, dof, expected = chi2_contingency(contingency_gender)

print(f"\nChi-Square Test Results:")
print(f"  Chi-square statistic: {chi2:.4f}")
print(f"  p-value: {p_value_freq_gender:.4f}")

if p_value_freq_gender < 0.05:
    print(f"REJECT H₀ - Significant difference in claim frequency between genders")
else:
    print(f"FAIL TO REJECT H₀ - No significant difference")

# Claim Severity (T-Test)
print("Claim Severity by Gender")


# Get claims data
claims_gender = df[df['Claimed'] == True]

female_severity = claims_gender[claims_gender['Gender'] == 'Female']['TotalClaims'].values
male_severity = claims_gender[claims_gender['Gender'] == 'Male']['TotalClaims'].values

print(f"\nSeverity Statistics:")
print(f"  Female: n={len(female_severity)}, Mean=R{female_severity.mean():.2f}, Median=R{np.median(female_severity):.2f}")
print(f"  Male: n={len(male_severity)}, Mean=R{male_severity.mean():.2f}, Median=R{np.median(male_severity):.2f}")

# T-test
t_stat, p_value_sev_gender = ttest_ind(female_severity, male_severity)

print(f"\nT-Test Results:")
print(f"  t-statistic: {t_stat:.4f}")
print(f"  p-value: {p_value_sev_gender:.4f}")

if p_value_sev_gender < 0.05:
    print(f"REJECT H₀ - Significant difference in claim severity between genders")
else:
    print(f"FAIL TO REJECT H₀ - No significant difference")

HYPOTHESIS 4: Gender Risk Differences
Claim Frequency by Gender

Contingency Table:
            Claimed  Not Claimed
Female         790    4348
Male           745    4117

Claim Frequency:
  Female: 15.38%
  Male: 15.32%
  Difference: 0.05 percentage points

Chi-Square Test Results:
  Chi-square statistic: 0.0021
  p-value: 0.9638
FAIL TO REJECT H₀ - No significant difference
Claim Severity by Gender

Severity Statistics:
  Female: n=790, Mean=R8560.80, Median=R6993.50
  Male: n=745, Mean=R8562.22, Median=R6861.00

T-Test Results:
  t-statistic: -0.0045
  p-value: 0.9964
FAIL TO REJECT H₀ - No significant difference


## Hypothesis Testing Results Summary

| Hypothesis | KPI | Control | Test | Test Used | p-value | Decision (α=0.05) |
|------------|-----|---------|------|-----------|---------|-------------------|
| **H₁: Provinces** | Claim Frequency | Amhara (13.96%) | Somali (17.48%) | Chi-Square | 0.0087 | **REJECT** ✓ |
| **H₁: Provinces** | Claim Severity | Amhara (R8,437) | Somali (R8,824) | t-test | 0.5095 | Fail to Reject |
| **H₂: Zip Codes** | Claim Frequency | 30002 (10.10%) | 50004 (16.88%) | Chi-Square | 0.0391 | **REJECT** ✓ |
| **H₂: Zip Codes** | Claim Severity | 30002 (R7,858) | 50004 (R10,774) | t-test | 0.0983 | Fail to Reject |
| **H₃: Zip Codes** | Margin | 30002 (R1,625) | 50004 (R668) | t-test | 0.0056 | **REJECT** ✓ |
| **H₄: Gender** | Claim Frequency | Female [?%] | Male [?%] | Chi-Square | [p-value] | [Decision] |
| **H₄: Gender** | Claim Severity | Female (R?) | Male (R?) | t-test | [p-value] | [Decision] |